> **Already run — no Kaggle session needed.** This is pure CPU counting over the train splits (<2 seconds for all 3 datasets), so it was run directly against the `brgc_kg_splits` data already on hand rather than asking you to spend GPU/session time on it. Output is saved as `dataset_bias_diagnostics.csv` in this folder and already folded into `revision_manuscript/H_dataset_bias_discussion.md`. Nothing to do here unless you want to re-verify it yourself.

# BRGC Revision — Notebook 06: Dataset-bias analysis (CPU)

**Purpose (Reviewer 1 Q2, Reviewer 3 c2).** Explain *why* the tail-frequency rule is so strong on CoDEx and weaker on FB15k-237, using structural diagnostics: tail determinism, relation functionality, reverse-triple share, and relation imbalance — not just the entropy value.

**No GPU.** Attach `brgc-kg-splits`, Run All (seconds). Can run locally. Send me `dataset_bias_diagnostics.csv`.

In [ ]:
# === Why is p(r|t) so strong? Dataset-bias diagnostics (CPU) ===
# Reviewer 1 (Q2) and Reviewer 3 (c2): don't just report entropy; explain WHY the tail nearly fixes the relation
# on CoDEx and far less on FB15k-237. This quantifies the structural causes.
from pathlib import Path
import numpy as np, pandas as pd, collections, math
INPUT_ROOT = Path("/kaggle/input"); OUT=Path("/kaggle/working"); OUT.mkdir(exist_ok=True)
def find_splits(name):
    for c in INPUT_ROOT.rglob(name):
        if c.is_dir() and all((c/s).exists() for s in ["train.txt","valid.txt","test.txt"]): return c
    return None
def load(p): return [tuple(l.strip().split("\t")) for l in open(p,encoding="utf-8") if l.strip()]

def H(counter):
    tot=sum(counter.values()); 
    if tot==0: return 0.0
    return -sum((n/tot)*math.log2(n/tot) for n in counter.values() if n>0)

rows=[]
for ds in ["CoDEx-S","CoDEx-M","FB15k-237"]:
    d=find_splits(ds)
    if not d: print("skip (not attached):",ds); continue
    tr=load(d/"train.txt")
    R=sorted({r for _,r,_ in tr}); 
    byt=collections.defaultdict(collections.Counter); byh=collections.defaultdict(collections.Counter)
    relcount=collections.Counter(r for _,r,_ in tr)
    for h,r,t in tr: byt[t][r]+=1; byh[h][r]+=1
    # 1) tail determinism: fraction of tails whose relations are a single relation
    tails=list(byt); single_tail=np.mean([len(byt[t])==1 for t in tails])
    # 2) conditional entropy H(R|T) weighted by tail frequency
    HT=sum(sum(c.values()) for c in byt.values())
    HRgT=sum((sum(byt[t].values())/HT)*H(byt[t]) for t in tails)
    HH=sum(sum(c.values()) for c in byh.values())
    HRgH=sum((sum(byh[h].values())/HH)*H(byh[h]) for h in byh)
    # 3) relation functionality: avg tails-per-(head,relation) ~ is r functional? (1 => functional)
    hr_tails=collections.defaultdict(set)
    for h,r,t in tr: hr_tails[(h,r)].add(t)
    functionality=np.mean([len(v) for v in hr_tails.values()])  # ~ how many tails per (h,r)
    # 4) inverse/reverse leakage: reverse triple (t,r,h) present for test pairs
    trainset=set(tr)
    reverse_share=np.mean([ (t,r,h) in trainset for (h,r,t) in tr[:20000] ])
    # 5) relation imbalance (top-1 relation share)
    top1=relcount.most_common(1)[0][1]/sum(relcount.values())
    rows.append([ds,len(R),round(HRgT,3),round(HRgH,3),round(single_tail,3),round(functionality,2),round(reverse_share,3),round(top1,3)])

df=pd.DataFrame(rows,columns=["Dataset","#relations","H(R|T) bits","H(R|H) bits",
    "frac tails w/ single relation","avg tails per (head,rel)","reverse-triple share","top-1 relation share"])
df.to_csv(OUT/"dataset_bias_diagnostics.csv",index=False)
print(df.to_string(index=False))
print("""
Reading the table:
- Low H(R|T) + high 'frac tails w/ single relation' => the tail entity almost fixes the relation
  (this is why p(r|t) is near the learned model on CoDEx).
- 'avg tails per (head,rel)' near 1 => many relations are near-functional, so the tail strongly implies r.
- FB15k-237 should show higher H(R|T), fewer single-relation tails, more candidates per tail => statistics help less.
This gives a structural CAUSE for the endpoint-statistics effect, not just the entropy number.""")